# 02b — Active inference versus Q-learning in a T-maze

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/02b_active_inference_vs_rl.ipynb)

Companion to §3.3 of *From Brains to Models*.

**Why this notebook.** Reinforcement learning explains much behaviour by assuming an animal chases an external reward. But a great deal of what animals do — sniffing, orienting, checking a signpost before committing — buys information rather than reward, and happens before any payoff is possible. Active inference reframes action as a way to make the world match the agent's own expectations, so that gathering information and satisfying preferences become one quantity. This notebook puts the two accounts side by side in a maze small enough to read line by line: one agent visits the cue on the first trial because knowing is valuable; the other must stumble onto reward before it can learn. Same four-location T-maze as the `pymdp` demo: a hidden context puts the reward in the left or right arm; the cue location reveals it without risk.

Active inference scores policies by expected free energy (pragmatic + epistemic). ε-greedy Q-learning sees only a scalar reward. The EFE loop is NumPy so the figure stays dependency-light; `pymdp` is the library you would use to scale this up.

**Claim.** When prediction acts, goals need not be an external reward: prior preferences and information gain can select the cue first.

**Method.** NumPy T-maze: expected-free-energy policy vs $\varepsilon$-greedy Q-learning. Seed 7.

**Result.** Active inference visits the cue on trial 1; Q-learning still samples.

**Limit.** Not a rat, not large-scale RL, not a claim that cortex runs this update.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **[Active inference](https://en.wikipedia.org/wiki/Active_inference).** A theory in which an agent acts to make its sensations match the world it expects to inhabit. "Goals" are built in as *preferred observations*, so seeking information and pursuing reward become two parts of one score.
- **Expected free energy (pragmatic + epistemic).** The quantity active inference minimises when choosing what to do next. A *pragmatic* part pulls toward preferred outcomes; an *epistemic* part rewards reducing uncertainty — the value of finding out.
- **Prior preference.** How strongly the agent wants each possible observation, fixed in advance; it plays the role a reward table plays in RL, but over outcomes rather than as a detached scalar.
- **[Reinforcement learning](https://en.wikipedia.org/wiki/Reinforcement_learning) and [Q-learning](https://en.wikipedia.org/wiki/Q-learning).** The comparison agent learns the value of each action from a scalar reward and, being $\varepsilon$-greedy, explores by acting at random a fraction of the time. It has no separate drive to gather information.
- **Cue (epistemic action).** Visiting the signpost yields no reward but removes uncertainty about which arm pays. Active inference values that visit; reward-only learning does not.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **[Active inference](https://en.wikipedia.org/wiki/Active_inference) / POMDP.** A discrete partially observable Markov process specified by matrices $A$ (likelihood $p(o\mid s)$), $B$ (transitions $p(s'\mid s,u)$), $C$ (log-preferences over observations), and $D$ (prior over the initial state).
- **Expected free energy $G(\pi)$.** At the grain of this notebook, $G(\pi)=\text{expected preference cost}-\text{expected information gain}$. Policies are chosen by $p(\pi)\propto\exp(-G(\pi))$ over a short horizon. Sign conventions differ across formulations; the intervention below does not.
- **[Q-learning](https://en.wikipedia.org/wiki/Q-learning).** $Q(s,a)\leftarrow Q(s,a)+\alpha\,[\,r+\gamma\max_{a'}Q(s',a')-Q(s,a)\,]$, with $\varepsilon$-greedy action selection.
- **Setup.** Four-location T-maze, horizon 2, seed 7; the expected-free-energy loop is NumPy (the `pymdp` library scales the same logic).
- **Failure condition.** Removing expected information gain should remove cue-seeking unless another exploration rule replaces it. The cue-first visit is therefore a test of the epistemic term, not evidence that cortex runs this update.

</details>

Writes `paper/figures/tmaze_aif_vs_ql.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.

### Concepts used in this notebook

This notebook runs two agents on the same T-maze and compares how they choose. The two blocks below define the recurring terms once — first in plain language, then in the notation used in the code. Both cover the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **T-maze task.** Four locations: a start, a cue that reveals which arm is baited, and two arms (one rewarding). The reward side is hidden until the cue is read.
- **Exploration versus exploitation.** Whether to gather information (visit the cue) or act on what is already known (head for the likely arm). The two agents resolve this tension differently.
- **Epistemic value (information gain).** The uncertainty an action is expected to remove; in active inference it is part of the objective, so curiosity needs no separate bonus.
- **Pragmatic value.** Expected progress toward preferred outcomes (here, reward) — the part both frameworks share.
- **Scalar reward.** A single number the reinforcement learner maximises; it says *how* good an outcome is but nothing about *why*.
- **What this is and is not.** A legible contrast between two update rules, not a model of a rat's hippocampus and not large-scale reinforcement learning.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **Generative model.** Matrices $A$ (likelihood), $B$ (transitions), $C$ (preferences), $D$ (initial prior); beliefs $q(s)$ are updated from observations by Bayesian filtering.
- **Policy scoring.** Expected free energy $G(\pi)$ over horizon $H=2$; a softmax over $-G(\pi)$ selects the next action.
- **Q-update.** Temporal-difference rule with learning rate $\alpha$, discount $\gamma$, and exploration rate $\varepsilon$.
- **Metrics.** The trial on which the cue is first visited, and cumulative reward over trials.
- **Implementation.** NumPy standing in for [active-inference](https://en.wikipedia.org/wiki/Active_inference) library `pymdp`; seed 7.
- **Limit.** A discrete POMDP; no claim that cortex runs this exact update, and no [viability](https://en.wikipedia.org/wiki/Allostasis) constraint beyond the designed preferences.

</details>

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.tmaze import (
    LOC_NAMES,
    render_paper_figure,
    run_experiment,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)

repo root: <checkout>
seed: 7


## The maze

Start at the center. The cue (bottom) tells you which arm is baited. Landing on the wrong arm is punished. An agent that values information goes to the cue *before* gambling on an arm.

In [2]:
traces = run_experiment(n_trials=40, n_steps=3, seed=SEED)
aif0, ql0 = traces["aif"][0], traces["ql"][0]
print("AIF trial 1:", [LOC_NAMES[i] for i in aif0.locations], "return", sum(aif0.rewards))
print("QL  trial 1:", [LOC_NAMES[i] for i in ql0.locations], "return", sum(ql0.rewards))
print(
    "cue-first rate  AIF", sum(t.visited_cue_first for t in traces["aif"]) / 40,
    " QL", sum(t.visited_cue_first for t in traces["ql"]) / 40,
)

AIF trial 1: ['center', 'cue', 'right', 'right'] return 2.0
QL  trial 1: ['center', 'right', 'cue', 'left'] return 0.0
cue-first rate  AIF 1.0  QL 0.575


## Paper figure

First-trial paths, then cumulative cue-first rate and mean return. Active inference is at ceiling from trial 1; Q-learning is still sampling.

In [3]:
fig = render_paper_figure(traces)
plt.show()
out = write_figure(traces, root / "paper" / "figures")
print("wrote", out)

/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49843/4235734862.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote <checkout>/paper/figures/tmaze_aif_vs_ql.pdf
wrote <checkout>/paper/figures/tmaze_aif_vs_ql.png
wrote <checkout>/paper/figures/tmaze_aif_vs_ql.pdf


## What to notice

- Exploration is not a bonus added to reward: it is the epistemic term in expected free energy.
- Goals are prior preferences over observations (prefer reward, avoid punish), not an external scalar.
- Q-learning can learn the same policy, but only after being punished. That is the inversion the paper cares about: in biology, value is a property of beliefs about staying viable.

For a fuller discrete-POMDP implementation see [`pymdp`](https://pymdp-rtd.readthedocs.io/) and Heins et al. (2022).

## Why this example is here

This notebook is not a tutorial on reinforcement learning, and it is not a model of a rat. It is a controlled demonstration of the one fact §3.3 of the paper needs: **prediction can act, and when it does, goals need not be an external reward.** Notebook 02 showed a network predicting a patch it already had. Here an agent predicts *what it will see if it moves*, and it moves so as to make preferred observations more likely — including the observation “I now know which arm is baited.”

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have never met active inference, POMDPs, or Q-learning. The two readings below are of the *same* T-maze. Nothing in the code is a ward protocol, and nothing is a proof that brains maximise expected free energy. The point is to make the inversion visible: information-seeking is not a bonus added to reward; it is a term in the same score that also values food.

### The maze in detail

Four locations: center, cue, left arm, right arm. A hidden **context** (left-baited or right-baited) is drawn each trial. The cue, at the bottom, reveals the context without risk. The wrong arm punishes; the right arm rewards. Three steps are enough to go center → cue → baited arm.

An agent that *values knowing* will visit the cue first, then take the indicated arm. An agent that only sees a scalar reward, and that explores by tossing a coin ($\varepsilon$-greedy), will often gamble on an arm before it has looked. That is the whole design. The figure is that difference on trial 1, then over forty trials.

**For the clinician and the neuroscientist.** This is *look at the test before you commit to the treatment arm*. The cue is a low-cost, informative observation — a history, a vital sign, a scan that will not itself cure or kill. The arms are irreversible enough to be costly if the choice is wrong. A good clinician buys the cue. A pure trial-and-error learner can get to the same policy, but only after being punished. Differential diagnosis is, in this schematic, expected information gain; jumping to a procedure is exploiting a prior that is still $1/2$–$1/2$.

**For the mathematically inclined.** The environment is a tiny POMDP. Hidden state: context $c\in\{L,R\}$ and location $\ell$. Observations: a cue channel (none / left / right) and a reward channel (neutral / reward / punish). Actions: move to a location. The active-inference agent has the generative model $P(o,c\mid \ell)$ and a belief $q(c)$; the Q-learner has a table $Q(\ell,m,a)$ with a cue-memory bit $m$ and never represents $c$ as such.

### What each agent is asked to do

**Active inference** scores a short policy $\pi$ (a sequence of locations) by **expected free energy** $G(\pi)$. In this notebook the implementation is the sum, along the imagined path, of

$$
G(\pi)\;=\;\sum_t \Big(-\underbrace{\mathbb{E}[C(o_t)]}_{\text{pragmatic}}-\underbrace{I\big(c;o_t\mid \pi\big)}_{\text{epistemic}}\Big),
$$

where $C$ is a preference over observations (like reward, dislike punish) and $I(c;o\mid\pi)$ is the information the next observation is expected to give about context. The agent *minimises* $G$: it wants preferred observations *and* a smaller posterior entropy on $c$. With a two-step horizon, “go to the cue, then the indicated arm” wins on trial 1, before any punishment has occurred. Beliefs $q(c)$ are updated by Bayes on the actual cue or outcome; the generative model itself is given, not learned.

**Q-learning** maximises expected discounted reward. The update is the usual temporal-difference rule

$$
Q(s,a)\;\leftarrow\;Q(s,a)+\alpha\big(r+\gamma\max_{a'}Q(s',a')-Q(s,a)\big),
$$

and exploration is $\varepsilon$-greedy: with probability $\varepsilon$ pick a random move. There is no term that says “the cue is valuable because it resolves $c$.” The cue becomes valuable only once the table has learned, by being rewarded and punished, that the memory bit $m$ predicts which arm pays.

**For the clinician and the neuroscientist.** Preferences $C$ are closer to *what I cannot afford to observe* (hypoxia, a wrong-site incision) than to a scalar score. The fish that “expects” water, in the paper’s line, is not pretending to have a reward function; it is a creature whose prior over observations is the shape of staying alive. Q-learning can imitate that policy only after enough punishments. Biology does not get a free replay of fatal trials.

**For the mathematically inclined.** $G$ as coded here is a receding-horizon, discrete stand-in for the expected-free-energy functional in Friston et al. (2017): a pragmatic term $\mathbb{E}_q[\ln P(o\mid C)]$ (or a linear $C^\top p(o)$) plus an epistemic term that is prior entropy minus expected posterior entropy on the hidden context. RL is recovered, in spirit, by dropping the epistemic term and letting $C$ encode reward. The Q-update is stochastic approximation of the Bellman optimality operator on a fully observed *memory-augmented* state $(\ell,m)$, not on $c$. $\varepsilon$ is a heuristic; it does not come from $I(c;o)$.

### How to read the four panels

**Panel A — first trial, active inference.**  
Start at the center, go to the cue, then the baited arm. The path is already the information-seeking policy. Expected free energy paid for the detour *in prospect*.

**Panel A′ — first trial, Q-learning.**  
The same maze, a table of zeros, $\varepsilon=0.25$. The first path wanders (in the committed figure: center → an arm → elsewhere). Return on that trial is luck, not a plan.

**For the clinician and the neuroscientist.** Panel A is a student who checks the indication before the procedure. Panel A′ is a student who starts the procedure and updates after the complication. Both can become competent. Only one is competent *before* the first complication.

**For the mathematically inclined.** A is $\arg\min_\pi G(\pi)$ with $q(c)=\mathrm{Unif}$ and horizon 2. A′ is $\varepsilon$-greedy on $Q\equiv 0$, hence nearly uniform on actions. The interesting event is that $G$ ranks the cue-first policies at the top *without* any nonzero $Q$.

**Panel B — cue-first rate.**  
The cumulative fraction of trials on which the agent visited the cue before either arm. Active inference is at $1$ from trial 1. Q-learning climbs as the table learns that the cue predicts the payoff.

**Panel C — mean return.**  
Cumulative average of reward minus punishment per trial. Same shape: ceiling for active inference, a slow approach for Q-learning.

**For the clinician and the neuroscientist.** B is “did they look at the informative test first?” C is “how often did they get the outcome they wanted?” A method can look good on C after enough harm; B asks whether the information was valued *as information*.

**For the mathematically inclined.** These are Monte Carlo traces, $n=40$, one seed (`SEED = 7`). They are not a regret bound. Changing $\varepsilon$, $\alpha$, or the horizon will move the Q-learning curves; the qualitative gap on trial 1 is the robust fact, because it does not depend on those hyperparameters. It depends on whether $I(c;o)$ is in the score.

### What this is not

It is not `pymdp` (that is the library you would use to scale the same discrete POMDP). It is not a claim that humans minimise $G$, or that Q-learning is a bad model of the basal ganglia, or that reinforcement learning cannot be given an exploration bonus that *looks* like the cue-first policy. It can: add an intrinsic reward for information gain and the two agents converge in behaviour. The paper’s point survives that tweak. In active inference the epistemic term is *the same objective* as the pragmatic term, and the “reward” was never in the world — it was a prior over observations. In RL, exploration is added, and value is a property of $r$.

Scale is stated in the table in §3.3: RL is large; active inference, as usually implemented, is small and discrete. This maze is the small end.

### What this has to do with being human

Notebook 01: prediction of other people’s tokens, no body. Notebook 02: prediction of one’s own input, still no action. This notebook: prediction that *moves*, with goals as expectations about what should be observed. That is the closest the paper has yet come to value as biologists mean it — not a scalar attached to a state, but a creature that acts to keep itself within the observations it can survive.

The later sections (architectures, dynamics, workspace, morals) ask what still fails to appear when the lower lane of the timeline becomes very good at scores like $Q$ or like next-token loss. If you can see, in four locations and a cue, that *wanting to know* can fall out of the same maths as *wanting the food*, then “goals are prior preferences” is more than an assertion.

Change `SEED`, $\varepsilon$, or the horizon and the Q-learning path will move. The claim should not: **same maze, same three steps; one score that pays for information, one that does not; value as a belief, or value learned only through punishment.**


### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. On trial 1, the [active-inference](https://en.wikipedia.org/wiki/Active_inference) agent visits the cue. What term in expected free energy paid for that detour, and what clinical habit is that term a schematic of?

2. [Q-learning](https://en.wikipedia.org/wiki/Q-learning) can acquire the same policy after punishment. Why is “after enough bruises” not a viable story for a body, even if the TD update is sound?

3. Drop the epistemic term. Who still goes to the cue, and what two ingredients were doing the work? (You can answer from the equations without re-running.)

4. Preferences $C$ are priors over observations, not a reward sitting in the world. What preferred observation would a hypoxic patient be acting to avoid, and how is that different from a preferred token?

5. The $Q$-table never represents context $c$. It represents (location, memory bit). When is that enough as an algorithm, and when does a clinician still need the hidden diagnosis?

6. Adding an exploration bonus to [RL](https://en.wikipedia.org/wiki/Reinforcement_learning) can imitate cue-first behaviour. Why does that tweak not erase the paper’s contrast between the two scores?

7. Horizon 1 versus horizon 2: what does a one-step lookahead fail to value that a two-step policy can?

8. Same maze, two objectives. What computational requirement is the cue-first policy a candidate for?

9. Even the active-inference agent has no metabolism. What would “value as a belief” still lack if $G$ were minimised in a machine with no body that can fail [homeostasis](https://en.wikipedia.org/wiki/Homeostasis)?

10. As usually implemented, active inference is small and discrete; RL is large. Does that fact change the trial-1 gap, or only what we can deploy?


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Friston, K. J., Daunizeau, J., and Kiebel, S. J. (2009).** [*Reinforcement learning or active inference?*](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0006421). *PLoS ONE* 4(7), e6421.
   - *(i) From brains to models.* Puts [active inference](https://en.wikipedia.org/wiki/Active_inference) and [reinforcement learning](https://en.wikipedia.org/wiki/Reinforcement_learning) on the same page: action can minimise surprise (or a bound on it) rather than maximise an external scalar. This notebook is that contrast in a four-location T-maze.
   - *(ii) What AI reveals about being human.* The inversion that matters for theme (ii): in active inference a goal is a prior preference over observations, closer to a constitutive expectation about remaining a viable agent than to a reward bit in a table.

2. **Sajid, N., Ball, P. J., Parr, T., and Friston, K. J. (2021).** [*Active inference: demystified and compared*](https://arxiv.org/abs/1909.10863). *Neural Computation* 33(3), 674–712. arXiv:1909.10863.
   - *(i) From brains to models.* A discrete-state comparison on an OpenAI gym baseline. [Active inference](https://en.wikipedia.org/wiki/Active_inference) agents explore in a Bayes-optimal way from expected free energy; reward is just another observation one can have a preference over — including the case of no reward at all. [Q-learning](https://en.wikipedia.org/wiki/Q-learning) needs the scalar.
   - *(ii) What AI reveals about being human.* The opened paper’s reward-free conditions are the human-relevant remainder: we act under preferences about our own observations, not only under an external score. The T-maze cue visit is the toy of that fact.

3. **Friston, K., et al. (2017).** [*Active inference: A process theory*](https://doi.org/10.1162/NECO_a_00912). *Neural Computation* 29(1), 1–49.
   - *(i) From brains to models.* The process theory Sajid et al. demystify: perception and action minimise the same free-energy functional; policies are scored by expected free energy (pragmatic + epistemic). This notebook’s horizon-2 agent is that split — information gain versus prior preference — on four locations.
   - *(ii) What AI reveals about being human.* Expected free energy is not a chat preference. It is a score over observations an agent expects to occupy. Without a generative model of a world that can kill it, the epistemic term has nothing human to be about.

4. **Levine, S. (2018).** [*Reinforcement learning and control as probabilistic inference: Tutorial and review*](https://arxiv.org/abs/1805.00909). arXiv:1805.00909.
   - *(i) From brains to models.* The formal relative: [RL](https://en.wikipedia.org/wiki/Reinforcement_learning) can be written as inference in a graphical model when rewards are recast as likelihoods. The paper’s point stands: recovering RL as a special case (drop the epistemic term; encode reward as preference) does not make the conceptual inversion disappear.
   - *(ii) What AI reveals about being human.* Control-as-inference is still usually an external objective. Theme (ii) is whether the preference is the agent’s existence or the experimenter’s scalar.

5. **Heins, C., et al. (2022).** [*pymdp: A Python library for active inference in discrete state spaces*](https://doi.org/10.21105/joss.04098). *Journal of Open Source Software* 7(73), 4098.
   - *(i) From brains to models.* The library this notebook stands in for: discrete-state [active inference](https://en.wikipedia.org/wiki/Active_inference) with explicit A, B, C, D matrices. The NumPy agent here is the same T-maze logic, small enough to read.
   - *(ii) What AI reveals about being human.* A usable library does not scale the claim to cortex. Discrete POMDPs are where the comparison is currently runnable. That limit belongs in the same sentence as the cue-first curve.

6. **Friston, K. (2010).** [*The free-energy principle: a unified brain theory?*](https://www.fil.ion.ucl.ac.uk/~karl/The%20free-energy%20principle%20A%20unified%20brain%20theory.pdf). *Nature Reviews Neuroscience* 11, 127–138. Author PDF.
   - *(i) From brains to models.* The parent claim: value (expected reward) and its complement, surprise, keep reappearing as the quantity being optimised. [Active inference](https://en.wikipedia.org/wiki/Active_inference) is that claim closed under action.
   - *(ii) What AI reveals about being human.* Unification, if it holds, is a claim about organisms that persist far from equilibrium. A Q-table has no such necessity. [Allostasis](https://en.wikipedia.org/wiki/Allostasis) is not an $\varepsilon$-greedy bonus.


### Exercises

1. **Drop the epistemic term.** Expected free energy is $-$pragmatic value $-$ information gain. Zero `_info_gain_one_channel` on a horizon-2 agent, then again on a horizon-1 agent. Who still visits the cue? What two ingredients were doing the work?

<!-- solution-box: 02b_1_drop_the_epistemic_term.py -->
<details>
<summary><strong>Show solution</strong> — Drop the epistemic term</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/02b_1_drop_the_epistemic_term.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.tmaze import ActiveInferenceAgent, TMaze, run_trial_aif
def cue_rate(agent, n=40):
    env = TMaze(seed=SEED)
    return sum(int(run_trial_aif(env, agent, n_steps=3).visited_cue_first) for _ in range(n)) / n

def no_ig(horizon):
    agent = ActiveInferenceAgent(horizon=horizon)
    agent._info_gain_one_channel = lambda *args, **kwargs: 0.0
    return agent

print(f"horizon 2, full EFE     cue-first {cue_rate(ActiveInferenceAgent(horizon=2)):.2f}")
print(f"horizon 2, no IG        cue-first {cue_rate(no_ig(2)):.2f}")
print(f"horizon 1, no IG        cue-first {cue_rate(no_ig(1)):.2f}")
print(
    "\nDropping the epistemic bonus is not enough if the horizon can still "
    "plan *cue then arm*: the cue remains a means to a later preferred "
    "observation. A one-step agent with no information gain has no such "
    "plan and stays put. Two ingredients, not one slogan. Still a "
    "four-location teaching model, not a rat."
)
```

</details>
<!-- /solution-box -->


2. **A noisy cue.** The environment and the agent both default to `cue_accuracy=1`. Set both to $0.6$ and compare cue-first rate and mean reward over 40 trials against the perfect-cue agent. When does “go to the cue first” stop being worth the step?

<!-- solution-box: 02b_2_a_noisy_cue.py -->
<details>
<summary><strong>Show solution</strong> — A noisy cue</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/02b_2_a_noisy_cue.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.tmaze import ActiveInferenceAgent, TMaze, run_trial_aif
def score(acc, n=40):
    env = TMaze(cue_accuracy=acc, seed=SEED)
    agent = ActiveInferenceAgent(cue_accuracy=acc)
    cues, rews = [], []
    for _ in range(n):
        tr = run_trial_aif(env, agent, n_steps=3)
        cues.append(tr.visited_cue_first)
        rews.append(sum(tr.rewards))
    return float(np.mean(cues)), float(np.mean(rews))

for acc in (1.0, 0.6, 0.5):
    cue, rew = score(acc)
    print(f"cue accuracy {acc:.1f}  cue-first {cue:.2f}  mean return {rew:+.2f}")
print(
    "\nAt 0.5 the cue is a coin flip and information gain collapses. The "
    "policy is then a bet on an arm. Epistemic value tracks how much an "
    "observation would change the context belief, not a virtue of looking. "
    "A noisy instrument in clinic is the same arithmetic: a test that does "
    "not change the posterior is not worth the step, and that is still not "
    "a rat."
)
```

</details>
<!-- /solution-box -->


3. **Q-learning needs the stumble.** ε-greedy Q-learning has no information-gain term. After 80 trials, compare its cue-first rate to active inference’s, and print $Q$ from the start state for cue vs the two arms. Has the cue become the greedy action? What is still missing even if it later does?

<!-- solution-box: 02b_3_ql_needs_the_stumble.py -->
<details>
<summary><strong>Show solution</strong> — Q-learning needs the stumble</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/02b_3_ql_needs_the_stumble.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.tmaze import (
    CENTER,
    CUE,
    LEFT,
    RIGHT,
    ActiveInferenceAgent,
    QLearningAgent,
    TMaze,
    run_trial_aif,
    run_trial_ql,
)
n = 80
env_a = TMaze(seed=SEED)
env_q = TMaze(seed=SEED + 1)
aif = ActiveInferenceAgent()
ql = QLearningAgent(seed=SEED + 2, epsilon=0.25)
aif_hits = sum(int(run_trial_aif(env_a, aif, n_steps=3).visited_cue_first) for _ in range(n))
ql_hits = sum(int(run_trial_ql(env_q, ql, n_steps=3).visited_cue_first) for _ in range(n))
q0 = ql.Q[CENTER, 0]
print(f"AIF cue-first over {n}: {aif_hits / n:.2f}")
print(f"QL  cue-first over {n}: {ql_hits / n:.2f}")
print(f"Q(start, unknown)  cue {q0[CUE]:+.2f}  left {q0[LEFT]:+.2f}  right {q0[RIGHT]:+.2f}")
print(f"greedy from start: {('center', 'cue', 'left', 'right')[int(np.argmax(q0))]}")
print(
    "\nActive inference visits the cue on the first trial. Q-learning "
    "has only the stumbles that ε-greedy happened to take, and a three-step "
    "return is a harsh credit assignment. Even if a longer budget made "
    "the cue greedy, the goal would still be an external scalar, not a "
    "preferred observation, and not a body that must keep going."
)
```

</details>
<!-- /solution-box -->
